# Nearest neighbors from geometry to a reusable estimator

**Lecture 13 · Notebook 00 · CMOR 438 / INDE 577**

Nearest-neighbor methods make a direct claim: observations that are close in a chosen representation should have similar responses. This notebook turns that sentence into mathematics, inspectable NumPy, a tested package interface, and visual evidence.

## How to use this notebook

Run the **core** cells in order. Pause at each checkpoint before revealing the next result. Practice cells ask you to change a metric, neighbor count, or weighting rule; extension cells examine consequences that do not fit in the live route. The notebook explains experiments, while reusable code lives in `src/rice_dsm/ml/neighbors.py` and its executable specifications live in `tests/test_nearest_neighbors.py`.

If imports fail, do not install packages from a code cell. Use the VS Code terminal at the repository root:

```text
uv sync --locked
uv run python scripts/setup_course.py
```

## Learning objectives

By the end, you should be able to:

1. define a metric and explain how representation and units determine neighborhoods;
2. derive uniform and distance-weighted kNN classification and regression;
3. explain how $k$ controls locality, bias, variance, storage, and prediction cost;
4. distinguish hyperparameters from fitted state in a professional estimator API; and
5. identify the unit, property, failure, differential, and integration tests this algorithm needs.

## Why this matters

kNN is valuable far beyond being an easy first classifier. It is a transparent local baseline, a retrieval primitive, an anomaly clue, and the geometric core of many recommendation systems. It also exposes engineering truths that complicated models can hide: scaling is part of the model, validation must choose hyperparameters, inference may be more expensive than fitting, and a similarity measure encodes domain judgment.

Historically, Fix and Hodges studied nearest-neighbor discrimination in 1951, and Cover and Hart analyzed nearest-neighbor classification in 1967. “Nonparametric” does **not** mean “has no choices” or “ignores irrelevant variables.” It means the model is not restricted to a fixed finite-dimensional parameterization as sample size grows. The training observations themselves remain fitted state.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from rice_dsm.paths import find_project_root

PROJECT_ROOT = find_project_root()
EXPECTED_ENVIRONMENT = PROJECT_ROOT / ".venv"

# A notebook kernel is a long-running Python process.  Its executable—not the
# terminal prompt—determines which packages this notebook imports.
print("Python version:    ", sys.version.split()[0])
print("Python executable: ", sys.executable)
print("Active prefix:     ", sys.prefix)
print("Repository root:   ", PROJECT_ROOT)
print("Using project .venv:", Path(sys.prefix).resolve() == EXPECTED_ENVIRONMENT.resolve())

if Path(sys.prefix).resolve() != EXPECTED_ENVIRONMENT.resolve():
    raise RuntimeError(
        "This notebook is not using the repository .venv. In VS Code choose "
        "Kernel > Select Another Kernel > Jupyter Kernel > Rice DSM. If it is "
        "missing, close the notebook, run `uv sync --locked` and "
        "`uv run python scripts/setup_course.py` in the repository terminal, "
        "then reopen the notebook."
    )

sns.set_theme(style="whitegrid", context="notebook")

## 1. Geometry is a modeling decision

A **metric** $d:\mathcal X\times\mathcal X\to[0,\infty)$ obeys nonnegativity, identity of indiscernibles, symmetry, and the triangle inequality. For numerical vectors, the Minkowski family is

$$
d_p(x,z)=\left(\sum_{j=1}^{d}|x_j-z_j|^p\right)^{1/p},\qquad p\ge 1.
$$

$p=1$ gives Manhattan distance and $p=2$ gives Euclidean distance. Choosing coordinates is just as important as choosing $p$. Centimeters versus meters, raw price versus log price, and pixels versus learned embeddings create different neighborhoods.

**Checkpoint.** A bird weighs $4$ kg and has a beak length of $45$ mm. Which coordinate dominates raw Euclidean distance? Is standardization automatically scientifically correct?

In [ ]:
from rice_dsm.ml.neighbors import euclidean_distances

# These four labeled observations are deliberately small enough to calculate by hand.
training_points = np.array([
    [0.0, 0.0],
    [1.0, 1.0],
    [3.0, 1.0],
    [4.0, 3.0],
])
training_labels = np.array(["circle", "circle", "triangle", "triangle"])
query = np.array([[2.5, 1.5]])

distances = euclidean_distances(query, training_points)[0]
distance_table = pd.DataFrame({
    "training_row": np.arange(len(training_points)),
    "x_1": training_points[:, 0],
    "x_2": training_points[:, 1],
    "label": training_labels,
    "distance_to_query": distances,
}).sort_values("distance_to_query")

distance_table

In [ ]:
# Draw the distance segments so the table and the geometry tell the same story.
fig, axis = plt.subplots(figsize=(8, 6))
for label, marker, color in [("circle", "o", "#00205B"), ("triangle", "^", "#C8102E")]:
    mask = training_labels == label
    axis.scatter(training_points[mask, 0], training_points[mask, 1],
                 s=120, marker=marker, color=color, label=label)
for point, distance in zip(training_points, distances, strict=True):
    axis.plot([query[0, 0], point[0]], [query[0, 1], point[1]],
              color="0.72", linewidth=1.2, zorder=0)
    midpoint = (query[0] + point) / 2
    axis.text(midpoint[0], midpoint[1], f"{distance:.2f}", fontsize=9)
axis.scatter(*query[0], s=180, marker="*", color="#F2A900", edgecolor="black",
             label="query")
axis.set(xlabel=r"$x_1$", ylabel=r"$x_2$", title="Euclidean distance defines the neighborhood")
axis.legend()
plt.show()

## 2. From a neighborhood to a prediction

Let the training set be $D=\{(x_i,y_i)\}_{i=1}^n$. Write $N_k(x)$ for the indices of the $k$ training points with smallest distance to query $x$.

For regression, uniform kNN estimates the local conditional mean:

$$
\widehat f_k(x)=\frac{1}{k}\sum_{i\in N_k(x)} y_i.
$$

For classification with classes $c\in\mathcal C$, it estimates class probabilities by vote proportions,

$$
\widehat P(Y=c\mid X=x)=\frac{1}{k}\sum_{i\in N_k(x)}\mathbf 1(y_i=c),
$$

then predicts the class with largest vote. An inverse-distance variant replaces each $1/k$ with normalized $w_i(x)=1/d(x,x_i)$. Exact matches need special handling: they alone receive vote mass, avoiding division by zero and preventing distant rows from diluting identical observations.

In [ ]:
from rice_dsm.ml.neighbors import (
    KNearestNeighborsClassifier,
    KNearestNeighborsRegressor,
)

# The public API stores hyperparameters at construction and learns state in fit.
classifier = KNearestNeighborsClassifier(n_neighbors=3, weights="uniform")
classifier.fit(training_points, training_labels)

neighbor_distances, neighbor_indices = classifier.kneighbors(query)
predicted_class = classifier.predict(query)[0]
class_probabilities = classifier.predict_proba(query)[0]

print("neighbor row indices:", neighbor_indices[0])
print("neighbor distances:  ", np.round(neighbor_distances[0], 3))
print("class order:         ", classifier.classes_)
print("vote proportions:    ", class_probabilities)
print("prediction:          ", predicted_class)

assert predicted_class == "triangle"
assert np.isclose(class_probabilities.sum(), 1.0)

In [ ]:
# The search is unchanged for regression; only aggregation of neighbor targets changes.
training_response = np.array([1.0, 1.5, 3.5, 4.0])
regressor = KNearestNeighborsRegressor(n_neighbors=3, weights="uniform")
regressor.fit(training_points, training_response)

regression_prediction = regressor.predict(query)[0]
expected = training_response[neighbor_indices[0]].mean()

print(f"local-average prediction: {regression_prediction:.3f}")
assert regression_prediction == expected

### Worked example: one search, two supervised tasks

The classifier and regressor share the distance calculation and ordered neighbor indices. They differ only in the target contract and aggregation rule. This is a useful software design boundary: factor shared mechanics without pretending the statistical outputs are interchangeable. A class vote proportion is not automatically a calibrated probability, and a local mean is unreliable where training coverage is sparse.

## 3. Scaling can change the answer

Suppose two features are temperature in degrees Celsius and annual income in dollars. Raw Euclidean distance will usually treat a one-dollar change as comparable to a one-degree change and let the larger numerical range dominate. A `StandardScaler` learns a mean and standard deviation **from training data only**. Fitting it before the split leaks information about the held-out distribution.

Scaling is not a ritual. It encodes the judgment that one training-standard-deviation shift is comparable across coordinates. Domain-specific transforms or metrics may be better.

In [ ]:
# Rescaling the first coordinate changes which observation is nearest.
raw_references = np.array([[0.0, 100.0], [3.0, 0.0]])
raw_query = np.array([[0.0, 0.0]])
scaled_references = raw_references.copy()
scaled_references[:, 0] *= 100.0

raw_order = np.argsort(euclidean_distances(raw_query, raw_references)[0])
scaled_order = np.argsort(euclidean_distances(raw_query, scaled_references)[0])

print("nearest under original units:", raw_order[0])
print("nearest after rescaling x_1:", scaled_order[0])
assert raw_order[0] != scaled_order[0]

## 4. What $k$ controls

$k$ is a hyperparameter, not a fitted coefficient. At $k=1$, the prediction surface can follow individual observations: low local bias and high variance. Increasing $k$ smooths the surface: variance often falls, while local bias can rise. Choosing $k$ on the test set turns the test set into validation data. The professional sequence is train → validation selection → one final test evaluation.

Odd $k$ does **not** guarantee no multiclass ties, and even binary ties can occur with distance weights. Therefore, a deterministic tie policy belongs in the API contract and tests.

In [ ]:
# A labeled simulation isolates the effect of k; it is not empirical evidence about a population.
rng = np.random.default_rng(438)
angles = rng.uniform(0, 2 * np.pi, 80)
radii = rng.uniform(0.2, 2.0, 80)
simulation_x = np.column_stack([radii * np.cos(angles), radii * np.sin(angles)])
simulation_y = (radii > 1.05).astype(int)

grid_x, grid_y = np.meshgrid(np.linspace(-2.2, 2.2, 150), np.linspace(-2.2, 2.2, 150))
grid = np.column_stack([grid_x.ravel(), grid_y.ravel()])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
for axis, k in zip(axes, [1, 7, 31], strict=True):
    model = KNearestNeighborsClassifier(n_neighbors=k).fit(simulation_x, simulation_y)
    surface = model.predict(grid).reshape(grid_x.shape)
    axis.contourf(grid_x, grid_y, surface, levels=[-0.5, 0.5, 1.5],
                  colors=["#DDE7F2", "#F8D7D4"], alpha=0.85)
    axis.scatter(simulation_x[:, 0], simulation_x[:, 1], c=simulation_y,
                 cmap="coolwarm", edgecolor="white", s=35)
    axis.set(title=f"k = {k}", xlabel=r"$x_1$", ylabel=r"$x_2$", aspect="equal")
fig.suptitle("Increasing k smooths a local decision rule", fontsize=15)
plt.show()

## 5. Professional practice: the package is the source of truth

The implementation is in `src/rice_dsm/ml/neighbors.py`, not hidden in this notebook. The notebook imports it, constructs experiments, and interprets evidence. The package owns validation, deterministic tie behavior, fitted state, and reusable prediction. Tests own tiny analytic examples and failure conditions.

This split prevents three common failures:

- a corrected function in one notebook but stale copies elsewhere;
- “works in this kernel” code that cannot be imported by a service or test;
- output cells being mistaken for reproducible verification.

In [ ]:
# Inspect the durable implementation and its executable specification locations.
implementation_path = PROJECT_ROOT / "src/rice_dsm/ml/neighbors.py"
test_path = PROJECT_ROOT / "tests/test_nearest_neighbors.py"

print("implementation:", implementation_path.relative_to(PROJECT_ROOT))
print("tests:         ", test_path.relative_to(PROJECT_ROOT))
print("implementation exists:", implementation_path.is_file())
print("test module exists:    ", test_path.is_file())

assert implementation_path.is_file()
assert test_path.is_file()

### What to test

1. **Unit test:** a $3$-$4$-$5$ triangle produces distance $5$.
2. **Analytic behavior:** a hand-computable vote or mean is returned.
3. **Property test:** neighbor distances are nondecreasing and probabilities sum to one.
4. **Failure test:** invalid $k$, nonfinite data, wrong shape, or prediction before fitting raises a useful exception.
5. **Differential test:** predictions agree with scikit-learn when metric, scaling, weights, and tie conventions agree.
6. **Integration test:** a clean notebook kernel imports the installed package rather than a local code copy.

Run the focused evidence from the terminal:

```text
uv run pytest tests/test_nearest_neighbors.py -q
uv run ruff check src/rice_dsm/ml/neighbors.py tests/test_nearest_neighbors.py
```

## Debugging clinic and common failure modes

| Symptom | Likely cause | First check |
| --- | --- | --- |
| Every prediction is the majority class | $k$ too large or representation weak | class balance and selected neighbors |
| Results change after converting units | features were not transformed as one fitted pipeline | preprocessing boundary |
| Training is fast but API latency is poor | exact search scans stored rows at inference | batch latency and index choice |
| Impossibly strong validation | scaler or $k$ used held-out evidence | split and fit order |
| Notebook import differs from tests | wrong VS Code kernel or package code copied locally | `sys.executable` and import path |

**Common failure:** “kNN has no training” is misleading. Fit validates and stores data, learns class order, and may fit preprocessing or an index. The cost is deferred, not absent.

## Guided practice

1. Compute the three nearest labels to $(2,1.5)$ by hand and verify the code.
2. Change `weights` to `"distance"`. Explain why the result changes or does not.
3. Add one exact duplicate query with a conflicting label. Write the API policy you would want before running it.
4. Replace Euclidean distance with Manhattan distance on paper. Which neighbor identities change?

In [ ]:
# PRACTICE: complete a hand-checkable weighted regression case before generalizing.
practice_x = np.array([[0.0], [2.0], [5.0]])
practice_y = np.array([0.0, 4.0, 10.0])
practice_query = np.array([[1.5]])

practice_model = KNearestNeighborsRegressor(
    n_neighbors=2,
    weights="distance",
).fit(practice_x, practice_y)
practice_prediction = practice_model.predict(practice_query)[0]

# TODO: derive this value from normalized inverse distances in your notes.
print("weighted prediction:", practice_prediction)
assert practice_prediction == 3.0

## Independent practice

Design a six-row dataset where 1-NN is correct on every training row but a held-out query is ambiguous. Draw it, state the metric and tie policy, then encode it as a new test. Your pull request should explain why the test protects a public behavior rather than merely increasing coverage.

## Extension: the curse of dimensionality and search systems

In high dimension, pairwise distances often concentrate: nearest and farthest points become less distinguishable, while the volume needed to populate a local neighborhood grows rapidly. Exact brute-force search costs roughly $O(nd)$ per query and stores $O(nd)$ feature values. Trees can help in favorable low-dimensional geometries; approximate nearest-neighbor indexes trade retrieval recall for latency and memory. Those are measured system choices, not transparent implementation details.

Try adding independent noise features to the ring simulation and graph the ratio of nearest to farthest distance. Then state what changed: the data representation, not the voting rule.

## Retrieval practice

Without looking back, answer:

1. What mathematical object does $N_k(x)$ denote?
2. Why is standardization part of the fitted model?
3. Which data may choose $k$?
4. Why is a class vote proportion not necessarily calibrated?
5. Which state must be serialized to reproduce a kNN prediction?

## Takeaway

kNN is a local prediction rule built from a representation, metric, neighborhood size, weighting rule, and aggregation rule. Its simplicity makes assumptions visible. Professional use still requires leakage-safe preprocessing, deterministic contracts, validation, failure tests, resource measurement, and support diagnostics.

## Further reading

- Cover and Hart, “Nearest Neighbor Pattern Classification” (1967), *IEEE Transactions on Information Theory*.
- [scikit-learn nearest-neighbors guide](https://scikit-learn.org/stable/modules/neighbors.html).
- [NumPy broadcasting documentation](https://numpy.org/doc/stable/user/basics.broadcasting.html).
- [Python data model documentation](https://docs.python.org/3/reference/datamodel.html).